In [14]:
# IMPORTS
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import text, create_engine

In [15]:
# LOAD VARIABLES
load_dotenv()


# CONFIG
TABLE_NAME = "deputy_details"
BASE_URL = ("https://dadosabertos.camara.leg.br/api/v2/deputados/")

In [16]:
#  FUNCTIONS

#GET DEPUTY ID
def get_deputy_ids(engine):

    query = text("""
        select
            distinct(id)
        from bronze.legislatures
        where 1=1
        limit 10000000
    """)

    with engine.connect() as conn:
        result = conn.execute(query)
        ids = [row[0] for row in result]

    return ids

# GET DEPUTY
def get_deputy_details(deputy_id):

    print(f"Collecting deputy {deputy_id}")

    url = f"{BASE_URL}{deputy_id}"

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        data = response.json()

        return data.get("dados", {})

    except Exception as e:
        print(f"Error deputy {deputy_id}: {e}")
        return {}


# SAVE PARQUET FUNCTION
def save_parquet(df, table_name):

    # Create directory
    path = f"data/raw/{table_name}"

    os.makedirs(path, exist_ok=True)

    # File path
    file_path = (
        f"{path}/{table_name}.parquet"
    )

    # Save parquet
    df.to_parquet(file_path,index=False)

    print(f"Parquet saved at: {file_path}")


In [20]:
dfs = []

#get engine details
engine = create_engine(os.getenv("DATABASE_CONNECTION"))

# 2. search for ids in the database
deputy_list = get_deputy_ids(engine)

print(f"Total deputies to process: {len(deputy_list)}")

# 3. loop through ids
for deputy_id in deputy_list:

    data = get_deputy_details(deputy_id)
    

    if data:
        df_normalized = pd.json_normalize(data)
        dfs.append(df_normalized)

# 4. concatenate all
if dfs:

    final_df = pd.concat(dfs, ignore_index=True)

    print(f"Total rows collected: {len(final_df)}")

    save_parquet(
        df=final_df,
        table_name=TABLE_NAME
    )

else:
    print("No data collected.")



Total deputies to process: 1561


KeyboardInterrupt: 

In [24]:
df = pd.read_parquet("../../data/raw/deputy_details/deputy_details.parquet")

In [25]:
display(df)

,id,uri,nomeCivil,cpf,sexo,urlWebsite,redeSocial,dataNascimento,dataFalecimento,ufNascimento,...,ultimoStatus.nomeEleitoral,ultimoStatus.gabinete.nome,ultimoStatus.gabinete.predio,ultimoStatus.gabinete.sala,ultimoStatus.gabinete.andar,ultimoStatus.gabinete.telefone,ultimoStatus.gabinete.email,ultimoStatus.situacao,ultimoStatus.condicaoEleitoral,ultimoStatus.descricaoStatus
0,74152,https://dadosabertos.camara.leg.br/api/v2/depu...,ISAIAS SILVESTRE,22898603600,M,NaN,[],1948-03-18,NaN,MG,...,ISAIAS SILVESTRE,NaN,NaN,NaN,NaN,NaN,NaN,Suplência,Suplente,NaN
1,179384,https://dadosabertos.camara.leg.br/api/v2/depu...,JOSÉ PINTO DE LUNA,09169201862,M,NaN,[],1965-03-05,NaN,CE,...,PINTO DE LUNA,NaN,NaN,NaN,NaN,NaN,NaN,Fim de Mandato,Efetivado,NaN
2,168034,https://dadosabertos.camara.leg.br/api/v2/depu...,JULIANO DE SOUZA RABELO,42454808153,M,NaN,[],1971-01-13,NaN,GO,...,CABO JULIANO RABELO,NaN,NaN,NaN,NaN,NaN,NaN,Suplência,Suplente,NaN
3,205234,https://dadosabertos.camara.leg.br/api/v2/depu...,LUIS FELIPE SILVA DE SOUZA,38487365272,M,NaN,[],1971-10-04,NaN,AM,...,FELIPE SOUZA,NaN,NaN,NaN,NaN,NaN,NaN,Fim de Mandato,Suplente,NaN
4,121948,https://dadosabertos.camara.leg.br/api/v2/depu...,ADRIANO ANTÔNIO AVELAR,50746553153,M,NaN,"[https://twitter.com/adrianodobaldy, https://w...",1969-09-06,NaN,GO,...,Adriano do Baldy,419,4,419,4,3215-5419,dep.adrianodobaldy@camara.leg.br,Exercício,Titular,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1556,160644,https://dadosabertos.camara.leg.br/api/v2/depu...,ESMERINO NERI BATISTA FILHO,03990842234,M,NaN,[],1952-07-11,NaN,PA,...,MIRIQUINHO BATISTA,NaN,NaN,NaN,NaN,NaN,NaN,Fim de Mandato,Titular,NaN
1557,141463,https://dadosabertos.camara.leg.br/api/v2/depu...,JOSÉ ABELARDO GUIMARÃES CAMARINHA,38233754820,M,NaN,[],1952-03-22,NaN,SP,...,ABELARDO CAMARINHA,NaN,NaN,NaN,NaN,NaN,NaN,Fim de Mandato,Titular,NaN
1558,74571,https://dadosabertos.camara.leg.br/api/v2/depu...,NELSON VICENTE PORTELA PELLEGRINO,24289612504,M,NaN,[],1960-12-27,NaN,BA,...,Nelson Pellegrino,NaN,NaN,NaN,NaN,NaN,NaN,Vacância,Titular,
1559,204486,https://dadosabertos.camara.leg.br/api/v2/depu...,CARLOS MAURO BENEVIDES FILHO,15336735191,M,NaN,"[https://twitter.com/mauro_bfilho, https://www...",1959-03-09,NaN,CE,...,Mauro Benevides Filho,731,4,731,7,3215-5731,dep.maurobenevidesfilho@camara.leg.br,Exercício,Titular,NaN
